# All you need to modify

In [49]:
# add new code here 

In [ ]:
import pandas as pd
import os



students_sheets = pd.read_excel("/Users/lgteam/Downloads/csd1718-3-20/stu.xlsx", sheet_name=None)

adults_sheets = "/Users/lgteam/Downloads/csd1718-3-20/adu.xlsx"

all_sheets = "/Users/lgteam/Downloads/csd1718-3-20/all.xlsx"

# Student Summary Statistics - Target # of Students Served (don't include Total)
target_values = [225,100,75,175]




/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


# Code Part (don't need to modify)

## 🍀Student Summary Statistics:


In [51]:
df_part_by_hour=  students_sheets['Participants By Hour Band']
df_part_by_hour.head()

,Participants By Hour Band,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Participant Count,Participant Count,Participant Count,Hour Band,Hour Band,Hour Band,Hour Band,Hour Band,Hour Band
4,Institution,Site,GradeLevel,0,Less Than 15,15-44,45-89,90-179,180-269


In [52]:
# Use the 3rd row (index 2) as column names
df_part_by_hour.columns = df_part_by_hour.iloc[4]

# Drop the first three rows and reset index
df_part_by_hour = df_part_by_hour.iloc[5:].reset_index(drop=True)

df_part_by_hour.head(10)

4,Institution,Site,GradeLevel,0,Less Than 15,15-44,45-89,90-179,180-269
0,"Community School District 17/18, NYCDOE 8039",Academy for College Prep & Career Exploration,06,1,5,20,7,0,0
1,NaN,NaN,07,11,33,1,0,0,0
2,NaN,NaN,08,16,26,5,3,0,0
3,NaN,NaN,09,36,44,22,3,1,0
4,NaN,NaN,10,34,37,18,9,4,0
5,NaN,NaN,11,36,44,14,10,9,0
6,NaN,NaN,12,33,37,9,8,4,0
7,NaN,NaN,13,49,3,1,0,0,0
8,NaN,NaN,NaN,0,0,0,1,1,0
9,NaN,Subtotal,Subtotal,216,229,90,41,19,0


In [53]:
# the average column from Daily Site Attendance Summary
daily_site_att = students_sheets['Daily Site Attendance Summary']

daily_site_att.columns = daily_site_att.iloc[2]  # Set 3rd row as header
daily_site_att = daily_site_att.iloc[3:]          # Drop first 3 rows
daily_site_att.columns.name = None                # Remove column index name
daily_site_att = daily_site_att.reset_index(drop=True)

daily_site_att = daily_site_att[['Total']]        # Keep only Total column

daily_site_att = daily_site_att.iloc[:-1]         # Drop last row
daily_site_att['Total'] = daily_site_att['Total'].str.extract(r'(\d+\.?\d*)')  # Keep number only


In [54]:
print(df_part_by_hour.columns)

Index(['Institution', 'Site', 'GradeLevel', '0', 'Less Than 15', '15-44',
       '45-89', '90-179', '180-269'],
      dtype='object', name=4)


In [55]:
all_cols = ['0', 'Less Than 15', '15-44', '45-89', '90-179', '180-269', '270+']
served_cols = ['Less Than 15', '15-44', '45-89', '90-179', '180-269', '270+']
plus15_cols = ['15-44', '45-89', '90-179', '180-269', '270+']
plus90_cols = ['90-179', '180-269', '270+']

# Filter each col list to only include columns that exist in the dataframe
existing_cols = df_part_by_hour.columns.tolist()
all_cols     = [c for c in all_cols     if c in existing_cols]
served_cols  = [c for c in served_cols  if c in existing_cols]
plus15_cols  = [c for c in plus15_cols  if c in existing_cols]
plus90_cols  = [c for c in plus90_cols  if c in existing_cols]

# Convert to numeric — this fixes the TypeError
df_part_by_hour[all_cols] = df_part_by_hour[all_cols].apply(
    pd.to_numeric, errors='coerce'
).fillna(0)  # <-- replaces any non-numeric/NaN with 0 instead of leaving as string

# Filter Subtotal rows
df_sub = df_part_by_hour[df_part_by_hour['Site'] == 'Subtotal']

# Build completely new dataframe
df_totals = pd.DataFrame({
    '[Total # Enrolled]': df_sub[all_cols].sum(axis=1),
    '[Total # Served]':   df_sub[served_cols].sum(axis=1),
    '[Total # 15+]':      df_sub[plus15_cols].sum(axis=1),
    '[Total # 90+]':      df_sub[plus90_cols].sum(axis=1)
})

# Insert as the first column (position 0)
df_totals.insert(0, '[Target # of students served]', target_values)

In [56]:
## the average column from Daily Site Attendance Summary
daily_site_att['Total'] = daily_site_att['Total'].astype(int)
daily_site_att = daily_site_att.rename(columns={'Total': 'Avg. # of Students Per Day'})
# Insert after [Total # Served] (now at position 2)
df_totals.insert(3, 'Avg. # of Students Per Day', daily_site_att['Avg. # of Students Per Day'].values)
print(df_totals)

    [Target # of students served]  [Total # Enrolled]  [Total # Served]  \
9                             225                 595               379   
18                            100                 184               142   
30                             75                 199               198   
39                            175                 670               663   

    Avg. # of Students Per Day  [Total # 15+]  [Total # 90+]  
9                           47            150             19  
18                          58            107             66  
30                          19             70              4  
39                          90            195              3  


In [57]:
##INSERT SCHOOL NAMES
# Create an ordered list of valid school names, skipping NaN and 'Subtotal'
school_names = [
    row['Site']
    for _, row in df_part_by_hour.iterrows()
    if pd.notna(row['Site']) and row['Site'] != 'Subtotal' and row['Site'] != 'Total'
]

school_names

# Add the actual Site names as the leftmost column
df_totals.insert(0, 'School', school_names)

# Create the final total row
total_row = pd.DataFrame(df_totals.iloc[:, 1:].sum()).T
total_row.insert(0, 'School', 'Total')

# Append the total row
df_totals = pd.concat([df_totals, total_row], ignore_index=True)

df_totals

,School,[Target # of students served],[Total # Enrolled],[Total # Served],Avg. # of Students Per Day,[Total # 15+],[Total # 90+]
0,Academy for College Prep & Career Exploration,225,595,379,47,150,19
1,Brooklyn Arts and Science Elementary School,100,184,142,58,107,66
2,Olympus Academy,75,199,198,19,70,4
3,PS 249,175,670,663,90,195,3
4,Total,575,1648,1382,214,522,92


In [58]:
# Create formatted display columns directly (no helper % columns)

df_totals['# of students 15+ hrs total (% of Target)'] = (
    df_totals['[Total # 15+]'].astype(int).astype(str) +
    " (" +
    (
        (df_totals['[Total # 15+]'] /
         df_totals['[Target # of students served]']) * 100
    ).round().astype(int).astype(str) +
    "%)"
)

df_totals['# of students 90+ hrs total (% of Target)'] = (
    df_totals['[Total # 90+]'].astype(int).astype(str) +
    " (" +
    (
        (df_totals['[Total # 90+]'] /
         df_totals['[Target # of students served]']) * 100
    ).round().astype(int).astype(str) +
    "%)"
)
df_totals = df_totals.drop(columns=['[Total # 15+]', '[Total # 90+]'])
df_totals = df_totals.reset_index(drop=True)
df_totals


,School,[Target # of students served],[Total # Enrolled],[Total # Served],Avg. # of Students Per Day,# of students 15+ hrs total (% of Target),# of students 90+ hrs total (% of Target)
0,Academy for College Prep & Career Exploration,225,595,379,47,150 (67%),19 (8%)
1,Brooklyn Arts and Science Elementary School,100,184,142,58,107 (107%),66 (66%)
2,Olympus Academy,75,199,198,19,70 (93%),4 (5%)
3,PS 249,175,670,663,90,195 (111%),3 (2%)
4,Total,575,1648,1382,214,522 (91%),92 (16%)


##🌷Family Component Summary Statistics (Last Column)

In [59]:

# Load the specific sheet for Attendance Hours, skipping the top 2 rows
df_hours = pd.read_excel(adults_sheets, sheet_name="Participant Attendance Hours", skiprows=2)

# Ensure "HoursPresent" is numeric
df_hours["HoursPresent"] = pd.to_numeric(df_hours["HoursPresent"], errors='coerce')

# Clean the ParticipantId: convert to string and remove any trailing '.0' in case pandas loaded it as a float
df_hours["ParticipantId"] = df_hours["ParticipantId"].astype(str).str.replace(r'\.0$', '', regex=True)

# # Filter for rows where HoursPresent is greater than 0
# df_active = df_hours[df_hours["HoursPresent"] > 0]

# Filter for rows where HoursPresent is > 0 AND the ParticipantId length is NOT exactly 9 digits
df_active = df_hours[(df_hours["HoursPresent"] > 0) & (df_hours["ParticipantId"].str.len() != 9)]

# Group by 'Site' and count unique adults using 'ParticipantId'
result = df_active.groupby("Site")["ParticipantId"].nunique().reset_index()

# Rename the column for better presentation
result.rename(columns={"ParticipantId": "Parents Served (Total)"}, inplace=True)

# Calculate the total sum of the 'Parents Served (Total)' column
total_parents = result["Parents Served (Total)"].sum()

# Append the total row to the dataframe (using loc is the cleanest way)
result.loc[len(result)] = {"Site": "Total", "Parents Served (Total)": total_parents}

# Display the final table
display(result)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,Site,Parents Served (Total)
0,Olympus Academy,22
1,PS 249,1
2,Total,23


##  🌸Participant Demographics - missing info

In [60]:
df_part_demo=  students_sheets['Participant Demographics']
df_part_demo.head()

,Participant Demographics,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15
0,Participant Demographics,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Institution,Site,Last Name,First Name,ParticipantID,State ParticipantID,Adult,Date Of Birth,School,Grade Level,Gender,Race/Ethnicity,English Learner Status,Lunch Status,Special Education Status,IDEA Disability Type
3,"Community School District 17/18, NYCDOE 8039",PS 249,Aayan,Jawad,203622063,NaN,No,2016-05-18 00:00:00,PS 249,04,Male,Race and Ethnicity Unknown,No,Free,No,Not Entered
4,"Community School District 17/18, NYCDOE 8039",PS 249,Abedin,Zafirah,4699028703,252925318,No,2016-04-26 00:00:00,PS 249,04,Female,Asian,No,Free,No,Not Entered


In [61]:
# Use the 3rd row (index 2) as column names
df_part_demo.columns = df_part_demo.iloc[2]

# Drop the first three rows and reset index
df_part_demo = df_part_demo.iloc[3:].reset_index(drop=True)

df_part_demo.head()
print(df_part_demo.dtypes)

2
Institution                 object
Site                        object
Last Name                   object
First Name                  object
ParticipantID               object
State ParticipantID         object
Adult                       object
Date Of Birth               object
School                      object
Grade Level                 object
Gender                      object
Race/Ethnicity              object
English Learner Status      object
Lunch Status                object
Special Education Status    object
IDEA Disability Type        object
dtype: object


In [62]:
def summarize_missing_by_school(df, columns_to_check, category_col="Site"):
    if category_col not in df.columns:
        raise ValueError(f"{category_col} not found in DataFrame columns")

    missing_site_rows = df[df[category_col].isna() | (df[category_col].astype(str).str.strip() == "")].copy()

    subset = df[columns_to_check + [category_col]].copy()
    subset_for_grouping = subset[subset[category_col].notna()].copy()
    subset_for_grouping[category_col] = (
        subset_for_grouping[category_col].astype(str).str.title()
    )

    ## Flagging missing values per col
    for col in columns_to_check:
        cleaned = subset_for_grouping[col].astype(str).str.strip()
        not_entered_mask = cleaned.str.lower() == "not entered"
        if col == "Gender":
            valid_gender = cleaned.str.title().isin(["Male", "Female", "Non-Binary"])
            subset_for_grouping[col + "_missing"] = ((~valid_gender) | not_entered_mask).astype(int)
        else:
            subset_for_grouping[col + "_missing"] = (
                subset_for_grouping[col].isna() | not_entered_mask
            ).astype(int)
## Validating Participant IDs
    pid  = df.loc[subset_for_grouping.index, "ParticipantID"].astype(str).str.strip()
    spid = df.loc[subset_for_grouping.index, "State ParticipantID"].astype(str).str.strip()
    valid_pid  = pid.str.match(r'^[12]\d{8}$')
    valid_spid = spid.str.match(r'^[12]\d{8}$')
    subset_for_grouping["ParticipantID_missing"]       = (~valid_pid).astype(int)
    subset_for_grouping["State ParticipantID_missing"] = (~valid_spid).astype(int)

    valid_osis = (pid == spid) & valid_pid & valid_spid
    subset_for_grouping["OSIS_missing"] = (~valid_osis).astype(int)

    missing_cols = (
        [col + "_missing" for col in columns_to_check]
        + ["ParticipantID_missing", "State ParticipantID_missing", "OSIS_missing"]
    )
    pivot = (
        subset_for_grouping
        .groupby(category_col)[missing_cols]
        .sum()
        .reset_index()
    )
    total_row = pd.DataFrame(pivot[missing_cols].sum()).T
    total_row[category_col] = "Total"
    pivot = pd.concat([pivot, total_row], ignore_index=True)

    all_missing_flags = df.copy()
    pid_all  = all_missing_flags["ParticipantID"].astype(str).str.strip()
    spid_all = all_missing_flags["State ParticipantID"].astype(str).str.strip()
    valid_pid_all  = pid_all.str.match(r'^[12]\d{8}$')
    valid_spid_all = spid_all.str.match(r'^[12]\d{8}$')
    valid_osis_all = (pid_all == spid_all) & valid_pid_all & valid_spid_all

    for col in columns_to_check:
        cleaned = all_missing_flags[col].astype(str).str.strip()
        not_entered_mask = cleaned.str.lower() == "not entered"
        if col == "Gender":
            valid_gender = cleaned.str.title().isin(["Male", "Female", "Non-Binary"])
            all_missing_flags[col + "_missing"] = ((~valid_gender) | not_entered_mask).astype(int)
        else:
            all_missing_flags[col + "_missing"] = (all_missing_flags[col].isna() | not_entered_mask).astype(int)

    all_missing_flags["ParticipantID_missing"]       = (~valid_pid_all).astype(int)
    all_missing_flags["State ParticipantID_missing"] = (~valid_spid_all).astype(int)
    all_missing_flags["OSIS_missing"]                = (~valid_osis_all).astype(int)

    # ← NEW: flag DOBs where year > 2023 (i.e. born after 2023, less than ~3 years old)
    # ✅ Works with datetime objects directly
    dob_parsed = pd.to_datetime(all_missing_flags["Date Of Birth"], errors="coerce")
    all_missing_flags["DOB_too_young"] = (dob_parsed.dt.year > 2023).astype(int)  # ← NEW

    flag_cols = (
        [col + "_missing" for col in columns_to_check]
        + ["ParticipantID_missing"]
    )
    total_missing_rows = all_missing_flags[all_missing_flags[flag_cols].sum(axis=1) > 0].copy()

    # ← NEW: also pull rows where DOB is suspiciously young, even if nothing else is missing
    young_dob_rows = all_missing_flags[all_missing_flags["DOB_too_young"] == 1].copy()  # ← NEW

    return pivot, missing_site_rows, total_missing_rows, flag_cols, young_dob_rows  # ← NEW

In [63]:
# # Should show 1 row with 2025
# print(young_dob_rows["Date Of Birth"].tolist())

In [64]:
def apply_missing_highlights(output_filename, total_missing_rows, flag_cols, young_dob_rows):
    red_fill  = PatternFill("solid", start_color="FF9999", end_color="FF9999")
    blue_fill = PatternFill("solid", start_color="9999FF", end_color="9999FF")

    flag_to_original = {
        fc: fc[: -len("_missing")]
        for fc in flag_cols
        if fc.endswith("_missing")
    }

    wb = load_workbook(output_filename)

    # --- Red highlights on Total Missing Info ---
    ws = wb["Total Missing Info"]
    header = {cell.value: cell.column for cell in ws[1]}
    for row_idx, (_, row) in enumerate(total_missing_rows.iterrows(), start=2):
        for flag_col, orig_col in flag_to_original.items():
            if orig_col in header and row.get(flag_col, 0) == 1:
                ws.cell(row=row_idx, column=header[orig_col]).fill = red_fill

    # --- Blue highlights on Young DOB sheet ---
    ws2 = wb["Young DOB"]
    header2 = {cell.value: cell.column for cell in ws2[1]}
    if "Date Of Birth" in header2:
        dob_col_idx = header2["Date Of Birth"]
        for row_idx in range(2, len(young_dob_rows) + 2):
            ws2.cell(row=row_idx, column=dob_col_idx).fill = blue_fill

    wb.save(output_filename)
    print("Highlights applied.")

 call the function here

In [65]:


columns_of_interest = ["Date Of Birth", "Grade Level", "Gender"]

missing_summary, missing_site_rows, total_missing_rows, flag_cols, young_dob_rows = summarize_missing_by_school(
    df_part_demo, columns_of_interest
)


## 🪻Site Summary Report

In [66]:
import pandas as pd
import numpy as np
import os



# Load the specific sheets and skip rows accordingly
df_act = pd.read_excel(all_sheets, sheet_name="Activity-Session Details", skiprows=2)
df_enr = pd.read_excel(all_sheets, sheet_name="Session Enrollment by Session", skiprows=2)
# Excel limits sheet names to 31 chars, so 'Summary' is cut off to 'Summa'
df_att = pd.read_excel(all_sheets, sheet_name="Daily Activity Attendance Summa", skiprows=4)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [67]:
# 2. Extract and format the Activity-Session Details
cols_act = ["Site", "Activity", "Session", "Days Scheduled", "Session Start Date"]
df_act = df_act[cols_act].copy()
# Convert "Days Scheduled" to numeric, converting errors/blanks to NaN
df_act["Days Scheduled"] = pd.to_numeric(df_act["Days Scheduled"], errors='coerce')
df_act["Session Start Date"] = pd.to_datetime(df_act["Session Start Date"], errors='coerce')
today = pd.Timestamp.today().normalize()

In [68]:
# 3. Extract and format Session Enrollment by Session
cols_enr = ["Site", "Activity", "Session", "Enrolled Count"]
df_enr = df_enr[cols_enr].copy()
# Convert to numeric and rename the column
df_enr["Enrolled Count"] = pd.to_numeric(df_enr["Enrolled Count"], errors='coerce')
df_enr.rename(columns={"Enrolled Count": "Enrolled Participant"}, inplace=True)

In [69]:
# 4. Extract and format Daily Activity Attendance Summary
cols_att = ["Site", "Activity", "Session", "Total"]
df_att = df_att[cols_att].copy()

# Function to extract the numeric value from the "Total" column string
def extract_average(val):
    if pd.isna(val):
        return np.nan
    val_str = str(val).replace('Average:', '').strip()
    try:
        return float(val_str)
    except ValueError:
        return np.nan
# load the numeric and rename column
df_att["Total"] = df_att["Total"].apply(extract_average)
df_att.rename(columns={"Total": "Average Daily Attendance"}, inplace=True)

In [70]:
# 5. Get a list of unique, valid sites
sites = df_act['Site'].dropna().unique().tolist()
# Filter out any aggregate rows like 'Total' or empty strings
sites = [s for s in sites if str(s).strip() != '' and not str(s).startswith('Total') and not str(s).startswith('Average')]

# 6. Create a dictionary to hold the merged DataFrame for each site
site_tables = {}

for site in sites:
    # Filter the three dataframes for the current site
    site_act = df_act[df_act['Site'] == site].copy()
    site_enr = df_enr[df_enr['Site'] == site].copy()
    site_att = df_att[df_att['Site'] == site].copy()

    # Merge Activity and Enrollment
    merged_site = pd.merge(site_act, site_enr, on=["Site", "Activity", "Session"], how="outer")

    # Merge the result with Attendance
    merged_site = pd.merge(merged_site, site_att, on=["Site", "Activity", "Session"], how="outer")

    merged_site = merged_site[~(merged_site['Session Start Date'] >= today)]

    # Delete the 'Session Start Date' column from the final table
    if 'Session Start Date' in merged_site.columns:
        merged_site = merged_site.drop(columns=['Session Start Date'])

    # Replace NaN (null) values with "-"
    merged_site = merged_site.fillna("-")

    # merged_site = merged_site.replace(r'^\s*$', '-', regex=True)

    # Store in our dictionary
    site_tables[site] = merged_site.sort_values(by=['Activity', 'Session'], ascending=True).reset_index(drop=True)

In [71]:
# 7. Print or preview the results
for site_name, final_df in site_tables.items():
    print(f"\n{'='*50}")
    print(f"Site: {site_name}")
    print(f"{'='*50}")
    # Display the first few rows for validation in Colab
    display(final_df.head())


Site: Academy for College Prep & Career Exploration


,Site,Activity,Session,Days Scheduled,Enrolled Participant,Average Daily Attendance
0,Academy for College Prep & Career Exploration,Architecture,Architecture,38,25,4.0
1,Academy for College Prep & Career Exploration,Badminton,Badminton,10,28,7.0
2,Academy for College Prep & Career Exploration,College Fieldtrip Workshop,College Fieldtrip Workshop,6,65,18.0
3,Academy for College Prep & Career Exploration,ELA,ELA,97,22,4.0
4,Academy for College Prep & Career Exploration,Earth & Space,Earth & Space Tutoring,76,13,4.0



Site: Brooklyn Arts and Science Elementary School


,Site,Activity,Session,Days Scheduled,Enrolled Participant,Average Daily Attendance
0,Brooklyn Arts and Science Elementary School,Academic Enrichment,Saturday Academy Session 1,5,41.0,22.0
1,Brooklyn Arts and Science Elementary School,Academic Enrichment,Saturday Academy Session 2 3rd garde,9,14.0,-
2,Brooklyn Arts and Science Elementary School,Academic Enrichment,Saturday Academy Session 2 4th Grade,35,11.0,-
3,Brooklyn Arts and Science Elementary School,Academic Enrichment,Saturday Academy Session 2 5th grade,9,4.0,-
4,Brooklyn Arts and Science Elementary School,Art/STEM,Art/STEM 1st-2nd,37,18.0,13.0



Site: Olympus Academy


,Site,Activity,Session,Days Scheduled,Enrolled Participant,Average Daily Attendance
0,Olympus Academy,College Access,College Access,176,184.0,6.0
1,Olympus Academy,Creative Society- Ms. Anna and Mr. Robert,Creative Society- Ms. Anna and Mr. Robert,35,184.0,6.0
2,Olympus Academy,Family Engagment,Parent engagement,1,185.0,-
3,Olympus Academy,Family Engagment 9/25/2025,Parent engagement,1,-,19.0
4,Olympus Academy,Family engagement 3/3/26,Family engagement,1,-,15.0



Site: PS 249


,Site,Activity,Session,Days Scheduled,Enrolled Participant,Average Daily Attendance
0,PS 249,Art With Assia 1st & 4th,Art With Assia,111,629.0,95.0
1,PS 249,Art With Assia 3rd & 5th,Arts With Assia 3rd & 5th,111,631.0,8.0
2,PS 249,Art With Assia K & 2nd,Art With Assia K & 2nd,110,631.0,11.0
3,PS 249,Community Service,Morning Leaders,1,7.0,-
4,PS 249,Community Service,Nursing Home Cheers,2,10.0,10.0





##🌈Full Report Download

In [72]:
# from google.colab import files
from openpyxl import load_workbook
from openpyxl.styles import PatternFill

output_filename = "Processed_Site_Tables.xlsx"

with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
    df_totals.to_excel(writer, sheet_name="Student Statistics", index=False)
    print(f"Successfully created sheet for: Student Statistics (Rows: {len(df_totals)})")

    result.to_excel(writer, sheet_name="Parents Served Summary", index=False)
    print(f"Successfully created sheet for: Parents Served Summary (Rows: {len(result)})")

    missing_summary.to_excel(writer, sheet_name="Missing Summary", index=False)
    print(f"Successfully created sheet for: Missing Summary (Rows: {len(missing_summary)})")

    missing_site_rows.to_excel(writer, sheet_name="Missing Site Info", index=False)
    print(f"Successfully created sheet for: Missing Site Info (Rows: {len(missing_site_rows)})")

    display_cols = [c for c in total_missing_rows.columns if not c.endswith("_missing") and c != "DOB_too_young"]
    total_missing_rows[display_cols].to_excel(writer, sheet_name="Total Missing Info", index=False)
    print(f"Successfully created sheet for: Total Missing Info (Rows: {len(total_missing_rows)})")

    young_display_cols = [c for c in young_dob_rows.columns if not c.endswith("_missing") and c != "DOB_too_young"]
    young_dob_rows[young_display_cols].to_excel(writer, sheet_name="Young DOB", index=False)
    print(f"Successfully created sheet for: Young DOB (Rows: {len(young_dob_rows)})")

    for site_name, final_df in site_tables.items():
        safe_sheet_name = (
            str(site_name)[:31]
            .replace(":", "").replace("/", "").replace("\\", "")
            .replace("?", "").replace("*", "")
        )
        final_df.to_excel(writer, sheet_name=safe_sheet_name, index=False)
        print(f"Successfully created sheet for: {safe_sheet_name} (Rows: {len(final_df)})")

# Apply highlights after file is fully closed
apply_missing_highlights(output_filename, total_missing_rows, flag_cols, young_dob_rows)

print(f"\nAll data saved to '{output_filename}'! Triggering download...")
# files.download(output_filename)

import os
print(os.path.abspath(output_filename))

Successfully created sheet for: Student Statistics (Rows: 5)
Successfully created sheet for: Parents Served Summary (Rows: 3)
Successfully created sheet for: Missing Summary (Rows: 5)
Successfully created sheet for: Missing Site Info (Rows: 0)
Successfully created sheet for: Total Missing Info (Rows: 1033)
Successfully created sheet for: Young DOB (Rows: 0)
Successfully created sheet for: Academy for College Prep & Care (Rows: 26)
Successfully created sheet for: Brooklyn Arts and Science Eleme (Rows: 35)
Successfully created sheet for: Olympus Academy (Rows: 29)
Successfully created sheet for: PS 249 (Rows: 33)
Highlights applied.

All data saved to 'Processed_Site_Tables.xlsx'! Triggering download...
/Users/lgteam/Documents/GitHub/weekly-update-report-tool/Processed_Site_Tables.xlsx
